# Cuaderno 3 · Una encuesta de verdad


**Curso:** Descripción y Visualización de Datos · Universidad Adolfo Ibáñez · 2026
**Profesor:** Naim Bro · **Ayudante:** Martín Castillo


---

Los cuadernos 1 y 2 fueron el ensayo. Este es el partido.

Vamos a trabajar con **458 respuestas reales** de estudiantes UAI, recogidas en 2022 y 2023 con
una encuesta parecida a la que ustedes acaban de responder. Es una base de verdad, con todo el
desorden que eso significa: gente que escribió su estatura en metros, gente que la escribió en
centímetros, gente que escribió `178?`, y alguien que en la pregunta *"¿a cuántos kilómetros vives
de la universidad?"* respondió **"Ñuñoa"**.

Ese desorden no es un defecto de esta base: **es la condición normal de los datos**. Aprender a
detectarlo y arreglarlo es, literalmente, el Sprint 2 del curso.

Al terminar vas a poder:

1. Cargar una base grande directamente desde una planilla de Google.
2. Describir una población: composición, promedios, comparaciones entre grupos.
3. **Detectar y corregir datos sucios**, dejando registro de lo que hiciste.
4. Presentar el resultado en gráficos que se entiendan solos.

> 🔴 Antes de empezar: **Archivo → Guardar una copia en Drive**.

---
## Parte 1 · Cargar los datos

La base vive en una planilla de Google. Igual que en el cuaderno 2, usamos el link terminado en
`export?format=csv`, que es la versión "descargable" de la planilla.

Puedes mirar la planilla original acá:
[abrir en Google Sheets](https://docs.google.com/spreadsheets/d/1zejmpH_mMpFfjFDrQfmOrGZKl5aN_JQZtDl7QlfdcuE/edit?gid=2111879632).

In [ ]:
url_encuesta <- "https://docs.google.com/spreadsheets/d/1zejmpH_mMpFfjFDrQfmOrGZKl5aN_JQZtDl7QlfdcuE/export?format=csv&gid=2111879632"

encuesta <- read.csv(url_encuesta)

dim(encuesta)     # dos números: filas y columnas

`dim()` devolvió dos números: **458 filas** (personas) y **18 columnas** (preguntas).

> 💡 Si esta celda falla, revisa que estés conectado a internet y que el link esté completo. Como
> respaldo, esta misma base está publicada acá:
> `https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/encuesta_uai.csv`

### Los nombres de las columnas

Como los datos vienen de un formulario, los nombres son las preguntas completas. Ilegibles:

In [ ]:
names(encuesta)

Los reemplazamos por nombres cortos. Deben ser **18**, y en el mismo orden que las columnas:

In [ ]:
names(encuesta) <- c("marca_temporal", "campus", "seccion", "edad", "estatura",
                     "alimentacion", "distancia_km", "sistema_operativo",
                     "duchas_semana", "covid", "deporte_obligatorio", "religion",
                     "op_aborto", "op_eutanasia", "op_matrimonio",
                     "op_cannabis", "op_educacion", "op_constitucion")

str(encuesta)

### El diccionario de variables

Esto es lo que mide cada columna. Escribir un diccionario así es **obligatorio** en la entrega del
Sprint 2: sin él, nadie —ni ustedes en dos meses— puede saber qué significa cada número.

| Variable | Pregunta original | Tipo |
|---|---|---|
| `campus` | ¿A qué campus perteneces? | categórica (Santiago / Viña) |
| `seccion` | ¿A qué sección perteneces? | numérica (es un código, no una cantidad) |
| `edad` | ¿Cuál es tu edad? | numérica |
| `estatura` | ¿Cuál es tu estatura en centímetros? | **texto, requiere limpieza** |
| `alimentacion` | ¿Qué estilo alimenticio tienes? | categórica |
| `distancia_km` | ¿A qué distancia de la universidad vives? | **texto, requiere limpieza** |
| `sistema_operativo` | Sistema operativo del celular | categórica |
| `duchas_semana` | ¿Cuántas veces te bañas a la semana? | categórica (incluye "8 o más") |
| `covid` | ¿Has sido contagiado de COVID-19? | categórica (Sí / No) |
| `deporte_obligatorio` | Postura frente al deporte obligatorio, de 1 a 5 | numérica (escala) |
| `religion` | ¿Con qué religión te identificas? | categórica |
| `op_*` | Grado de acuerdo con seis temas públicos | categórica ordinal |

> ⚠️ Fíjate en `seccion`: R la leyó como número, pero **no es una cantidad**. Calcular "la sección
> promedio" (5,4) no significa nada. Que algo sea un número no lo convierte en numérico.

---
## Parte 2 · ¿Quiénes contestaron?

Lo primero, siempre: describir a la población de la muestra. `table()` cuenta cuántos hay de cada
categoría.

In [ ]:
table(encuesta$campus)

In [ ]:
table(encuesta$alimentacion)

En porcentajes se lee mucho mejor:

In [ ]:
round(prop.table(table(encuesta$alimentacion)) * 100, 1)

Un dato con el que siempre se sorprende el curso —la distribución de sistemas operativos:

In [ ]:
round(prop.table(table(encuesta$sistema_operativo)) * 100, 1)

Y la religión, ordenada de mayor a menor con `sort()`:

In [ ]:
sort(table(encuesta$religion), decreasing = TRUE)

### Cruzar dos variables

Con dos variables adentro, `table()` arma una **tabla de contingencia**. Es el primer paso para
comparar grupos.

In [ ]:
table(encuesta$campus, encuesta$sistema_operativo)

Pero esa tabla engaña: Santiago tiene 344 respuestas y Viña 114, así que los conteos no se pueden
comparar directamente. Hay que pasar a porcentajes **dentro de cada campus** — eso hace
`margin = 1` ("calcula los porcentajes por fila"):

In [ ]:
round(prop.table(table(encuesta$campus, encuesta$sistema_operativo), margin = 1) * 100, 1)

> 🔑 Este es uno de los errores más comunes en visualización de datos, y lo van a ver en la clase 9:
> **comparar conteos entre grupos de tamaños distintos**. Casi siempre hay que normalizar por
> el tamaño del grupo antes de comparar.

---
## Parte 3 · El corazón de la clase: la estatura

Vamos con una pregunta simple: **¿cuánto mide en promedio un estudiante UAI?**

In [ ]:
mean(encuesta$estatura)

Fíjate bien: R **no** se cayó. Devolvió `NA` —su forma de decir "dato faltante"— junto con una
advertencia: `argument is not numeric or logical`, es decir *"me pediste el promedio de algo que no
es un número"*.

> 🧠 **La primera lección del día:** que el código corra no significa que el resultado sirva. Un
> `NA` o un promedio absurdo pasan igual de silenciosos que un buen resultado.

Miremos qué contiene la columna:

In [ ]:
head(encuesta$estatura, 25)

Las **comillas** alrededor de cada valor son la pista: R está guardando esto como *texto*. Y basta
que una sola persona haya escrito `163cm` para que **toda la columna** se vuelva texto.

Veamos exactamente cuáles son los casos con letras u otros símbolos:

In [ ]:
unique(encuesta$estatura[grepl("[^0-9.,]", encuesta$estatura)])

Ahí están: `1,78 cm`, `163cm`, `179 cm`, `177cm`... y un memorable `178?`.

### Paso 1: convertir a número

La receta, de adentro hacia afuera:

1. `gsub("[^0-9.,]", "", x)` → borra todo lo que **no** sea dígito, punto o coma.
2. `gsub(",", ".", x)` → cambia la coma decimal por punto (R sólo entiende el punto).
3. `as.numeric(x)` → convierte el texto a número.

In [ ]:
estatura_num <- encuesta$estatura
estatura_num <- gsub("[^0-9.,]", "", estatura_num)   # fuera letras, símbolos y espacios
estatura_num <- gsub(",", ".", estatura_num)         # coma  ->  punto
estatura_num <- as.numeric(estatura_num)             # texto ->  número

summary(estatura_num)

### Paso 2: leer el `summary()` con desconfianza

Mira esos números con calma. Hay tres cosas mal:

- **Mínimo = 1.53** → alguien de metro y medio... o alguien que contestó en **metros**.
- **Máximo = 15800** → nadie mide 158 metros. Es un `158` con dos ceros de más.
- **`NA's = 4`** → cuatro personas dejaron la pregunta en blanco. `NA` es como R representa un
  dato faltante, y hay que decidir explícitamente qué hacer con él.

La media que reporta —161.5— **no es la estatura promedio de nadie**: es el promedio de una mezcla
de metros, centímetros y un error de tipeo.

> 🧠 **La lección de la clase:** un cálculo que corre sin error no es un cálculo correcto. Antes de
> reportar un promedio, siempre mira el mínimo, el máximo y la cantidad de `NA`.

Un gráfico lo deja aún más claro:

In [ ]:
boxplot(estatura_num,
        main = "Estatura declarada, SIN limpiar",
        ylab = "Valor declarado")

El punto solitario allá arriba (el 15800) aplasta todo el resto del gráfico: no se ve nada.

### Paso 3: metros → centímetros

Regla: si el valor es menor que 3, la persona contestó en metros. Lo multiplicamos por 100.

Los **corchetes** `[ ]` seleccionan sólo algunos elementos de un vector, y `which()` entrega
**las posiciones donde se cumple una condición**:

In [ ]:
sum(estatura_num < 3, na.rm = TRUE)     # ¿cuántas personas contestaron en metros?

> El `na.rm = TRUE` significa *"ignora los datos faltantes al calcular"*. Sin él, cualquier `NA`
> contagia el resultado y R devuelve `NA`. Lo vas a escribir muchas veces este semestre.

In [ ]:
estatura_num[which(estatura_num < 3)] <- estatura_num[which(estatura_num < 3)] * 100

summary(estatura_num)

### Paso 4: los valores imposibles

Ya no hay metros, pero el máximo sigue en 15800 y el mínimo bajó a 58. Ninguna de las dos es una
estatura humana posible.

Aquí hay que **tomar una decisión y dejarla escrita**: consideraremos válidas las estaturas entre
**120 y 220 cm**, y marcaremos el resto como dato faltante (`NA`).

No es la única decisión razonable —se podría corregir el 15800 a 158, por ejemplo—, pero sí tiene
que ser **explícita**. Eso es trazabilidad, uno de los criterios transversales del curso.

In [ ]:
sospechosos <- estatura_num[estatura_num < 120 | estatura_num > 220]
sospechosos[!is.na(sospechosos)]       # veamos exactamente qué vamos a descartar

In [ ]:
estatura_num[estatura_num < 120 | estatura_num > 220] <- NA

encuesta$estatura_cm <- estatura_num   # la guardamos como columna nueva

summary(encuesta$estatura_cm)

Ahora sí: mínimo 150, máximo 198, promedio 173 cm. Números que describen personas reales.

Los `NA` pasaron de 4 a 6: los 4 que venían en blanco, más los 2 que acabamos de descartar. Esa
diferencia es justamente lo que hay que poder explicar después.

Y el gráfico por fin muestra algo:

In [ ]:
hist(encuesta$estatura_cm,
     main = "Distribución de la estatura (datos limpios)",
     xlab = "Estatura (cm)",
     ylab = "Número de estudiantes",
     col  = "steelblue")

> 📋 **Para la bitácora.** Todo lo que acabamos de hacer se resume en cuatro decisiones que
> cualquiera podría repetir:
> 1. Se eliminaron letras y símbolos de la respuesta.
> 2. Se interpretó la coma como separador decimal.
> 3. Los valores menores a 3 se interpretaron como metros y se multiplicaron por 100 (122 casos).
> 4. Los valores fuera del rango 120–220 cm se marcaron como faltantes (2 casos: 58 y 15800).
>
> Resultado: 452 estaturas válidas de 458 respuestas.
>
> Ese párrafo —no el código— es lo que hace que un análisis sea creíble.

---
## Parte 4 · La distancia a la universidad

Misma historia, versión con más creatividad. Miremos las respuestas que traen algo distinto de un
número:

In [ ]:
head(unique(encuesta$distancia_km[grepl("[^0-9.,]", encuesta$distancia_km)]), 20)

`10 km`, `22.5 KM`, `11km aprox`, `13 kilómetros`, `2 o 1`... y **`Ñuñoa`**, que es una comuna, no
una distancia.

La misma receta se hace cargo de casi todo. Fíjate en qué pasa con los casos irreductibles:

In [ ]:
distancia <- encuesta$distancia_km
distancia <- gsub("[^0-9.,]", "", distancia)
distancia <- gsub(",", ".", distancia)
distancia <- as.numeric(distancia)

summary(distancia)

Mira el final del `summary()`: **12 `NA`**. Son los casos donde, después de borrar las letras, no
quedó ningún número que convertir: `Ñuñoa` y las respuestas en blanco.

Por un lado eso es exactamente lo que queremos: **lo que no se puede interpretar se marca como
faltante, no se inventa.** Por otro lado, fíjate en el peligro: **R lo hizo en completo silencio**,
sin errores ni advertencias. Si no miras el `summary()`, pierdes 12 casos sin enterarte.

> ⚠️ Cuidado con `2 o 1`: la receta lo convirtió en `21`, un número perfectamente plausible y
> completamente falso. Ninguna limpieza automática es perfecta; por eso siempre hay que mirar los
> casos raros a mano.

In [ ]:
encuesta$distancia <- distancia

hist(encuesta$distancia,
     main = "¿A qué distancia de la universidad viven?",
     xlab = "Kilómetros",
     ylab = "Número de estudiantes",
     col  = "darkorange")

In [ ]:
round(median(encuesta$distancia, na.rm = TRUE), 1)   # la mediana resiste mejor los valores extremos

---
## Parte 5 · Las opiniones

La encuesta preguntó el grado de acuerdo con seis temas. Miremos uno:

In [ ]:
table(encuesta$op_aborto)

Funciona, pero las categorías salen en orden alfabético —`De acuerdo`, `En desacuerdo`, `Ni de
acuerdo ni en desacuerdo`—, que no es el orden lógico de una escala de opinión.

Para arreglarlo convertimos la variable en un **factor**, que es el tipo de dato de R para
categorías, y le decimos explícitamente en qué orden van sus niveles:

In [ ]:
escala <- c("En desacuerdo", "Ni de acuerdo ni en desacuerdo", "De acuerdo")

encuesta$op_aborto <- factor(encuesta$op_aborto, levels = escala)

table(encuesta$op_aborto)

Ahora el orden tiene sentido y el gráfico se lee de izquierda a derecha como una escala:

In [ ]:
barplot(table(encuesta$op_aborto),
        main = "Grado de acuerdo con el aborto libre",
        ylab = "Número de estudiantes",
        col  = c("tomato", "gray80", "steelblue"),
        names.arg = c("En desacuerdo", "Ni/Ni", "De acuerdo"))

> 🎨 Nota el detalle del color: rojo y azul en los extremos, gris en el medio. El color aquí no es
> decorativo, **codifica información**. Volveremos sobre esto en la clase 9.

Nota también que 4 personas dejaron la pregunta en blanco y no aparecen en el gráfico. Cuando
reportes un porcentaje, siempre di sobre qué total lo calculaste.

---
## Parte 6 · Comparar grupos

Describir es interesante; **comparar** lo es más. Tres formas de hacerlo.

### 1. Una variable numérica entre grupos: boxplot

La virgulilla `~` se lee **"según"**.

In [ ]:
boxplot(estatura_cm ~ campus,
        data = encuesta,
        main = "Estatura según campus",
        xlab = "Campus",
        ylab = "Estatura (cm)",
        col  = c("lightblue", "lightgreen"))

### 2. Promedios por grupo: `tapply()`

`tapply(variable, grupo, función)` aplica una función a cada grupo por separado.

In [ ]:
tapply(encuesta$estatura_cm, encuesta$campus, mean, na.rm = TRUE)

In [ ]:
# Postura frente al deporte obligatorio (escala 1 a 5), según campus
round(tapply(encuesta$deporte_obligatorio, encuesta$campus, mean, na.rm = TRUE), 2)

### 3. Dos variables categóricas: tabla de porcentajes

¿Se relaciona la religión declarada con la opinión sobre el aborto libre? Cruzamos y calculamos
porcentajes por fila:

In [ ]:
tabla <- table(encuesta$religion, encuesta$op_aborto)

round(prop.table(tabla, margin = 1) * 100, 1)

> ⚠️ **Cuidado al leer esta tabla.** Varias filas tienen muy pocos casos (`Islam` tiene 2, `Judía`
> tiene 5). Un "100%" calculado sobre 2 personas no dice nada sobre esa población: dice algo sobre
> esas dos personas.
>
> Mostrar porcentajes sin mostrar el tamaño del grupo es una de las formas más frecuentes —y más
> difíciles de detectar— de mentir con datos. Cuando reportes porcentajes por grupo, reporta
> siempre el `n`:

In [ ]:
table(encuesta$religion)     # el n de cada grupo, para leer la tabla anterior con criterio

---
## Parte 7 · ✏️ Tu turno

Trabaja en parejas. Agrega las celdas que necesites con **+ Código**.

1. ¿Qué porcentaje del curso declara haber tenido COVID-19?
2. ¿Cuál es la edad promedio y la edad mediana? ¿Por qué son distintas?
   (Pista: mira `max(encuesta$edad)` y piensa qué le hace ese valor a cada medida.)
3. La variable `duchas_semana` **parece** numérica pero no lo es: contiene `"8 o más"` y
   `"Prefiero no responder"`. Haz una `table()` y decide, argumentando, qué harías con esas dos
   categorías si necesitaras calcular un promedio.
4. Elige otra de las variables de opinión (`op_eutanasia`, `op_matrimonio`, `op_cannabis`,
   `op_educacion`, `op_constitucion`), ordénala como factor y grafícala.
5. **Comparación libre:** elige dos variables que creas que se relacionan, muéstralo con una tabla
   o un gráfico, y escribe en una **celda de texto**: qué encontraste, y qué **no** puedes concluir
   a partir de eso.

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.

In [ ]:
# 4.

In [ ]:
# 5.

---
## Cierre: lo que realmente pasó hoy

Cuenta las líneas de este cuaderno. Las que calculan promedios y hacen gráficos son pocas; la
mayoría fueron para **entender y arreglar los datos**.

Esa proporción no es un accidente de esta clase: es la profesión. En cualquier equipo de datos, la
mayor parte del tiempo se va en conseguir, limpiar, verificar y documentar la información. El
gráfico bonito es la última milla.

Por eso el proyecto del semestre tiene un sprint completo (**Sprint 2 · prototipo de datos**)
dedicado exactamente a esto: base limpia, diccionario de variables, decisiones documentadas.

### Las tres preguntas que deberías hacerle a cualquier base de datos

1. **¿De dónde vienen estos datos y quién quedó fuera?** (acá: sólo estudiantes que quisieron
   responder, de dos campus, en 2022-2023)
2. **¿Qué mide realmente cada variable?** (acá: estatura *declarada*, no medida)
3. **¿Qué tuve que decidir para poder calcular algo?** (acá: metros vs. centímetros, rango válido,
   qué hacer con los `NA`)

Si sabes responder esas tres preguntas sobre los datos de tu proyecto, vas bien.

---

### Lo que viene

- **Clase 3:** dominios, preguntas y fuentes de datos → parte el Sprint 1.
- **Clases 5 y 6:** `dplyr`, que convierte todo lo de hoy en código mucho más legible.
- **Clase 8:** `ggplot2`, y ahí los gráficos empiezan a verse en serio.

**¡Nos vemos la próxima clase!** 🚀